In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from time import time, sleep
from tqdm import tqdm

import pyvisa

from Code.device_managers import Devices
from Code.experimentA_manager import ExperimentA
from Code.experimentB_manager import ExperimentB

In [2]:
from Code.experiment_configs import Debug_Config, Test_Config, Final_Config

CONFIG = Final_Config()

CONFIG.summary()

Expected runtime of A: 189.01666666666668 min
Expected runtime of A: 3.1502777777777777 hr
Expected runtime of B: 779.6666666666666 min
Expected runtime of B: 12.994444444444444 hr

Total runtime is at least: 968.6833333333333 min
Total runtime is at least: 16.14472222222222 hr

Will save to: Data/Final


In [3]:
devices = Devices()
devices.reset()
devices.write("AMPL", 10.0)

In [4]:
# len in meters
d = 200e-6
l = 0.02

# N / m**2
E = 98e9
# kg / m**3
rho = 8.5e3
v_upsilon = np.sqrt(E / rho)

ALPHA_N = [1.424987, 0.992249, 1.000198, 0.999994]

def get_alpha_n(n: int):
    if n > 3:
        return 1
    return ALPHA_N[n]

def calc_free_reed_eigenfrequency(n: int):
    alpha_n = get_alpha_n(n)
    nu = alpha_n * (2 * n + 1)**2 * np.pi * d * v_upsilon / (16 * np.sqrt(3) * l**2)
    return nu

target_freqs = [calc_free_reed_eigenfrequency(n)/2 for n in range(4)]
print(target_freqs)
for i, freq in enumerate(target_freqs[1:]):
    print(f"nu_{i+1}/nu_0 = {freq / target_freqs[0]}")

theoretical_target_freqs = target_freqs[:3]


[137.12724434345407, 859.361762354977, 2406.2394172338777, 4715.267337470035]
nu_1/nu_0 = 6.266892961128769
nu_2/nu_0 = 17.547493415729406
nu_3/nu_0 = 34.38607229399286


In [7]:
trials_at_each_freq = CONFIG.A_trials_at_each_freq
num_of_freqs = CONFIG.A_num_of_freqs
# determined padding of freq range around theoretical target freq
pads = [4, 7, 10]

loc_prefix = CONFIG.loc_prefix

target_freqs = [
    125,
    825,
    2406.2394172338777
]

devices = Devices()
devices.reset()
sleep(CONFIG.A_temp_initializing)

print(f"Analysing {len(target_freqs)} resonance peaks.")
for i, (freq, theo_freq, pad) in enumerate(zip(target_freqs, theoretical_target_freqs, pads)):
    loc = f"{loc_prefix}/Experiment_A/resonance{i}_data"
    freqs_series = np.geomspace(freq - pad, freq + pad, num=num_of_freqs)
    expA = ExperimentA(loc, freqs_series, trials=trials_at_each_freq, init_metadata={"target_freq": freq, "theoretical_target_freq": theo_freq})
    
    expA.execute()

Analysing 3 resonance peaks.
Starting Experiment with expected runtime: 3.6 s.


100%|██████████| 2/2 [00:05<00:00,  2.82s/it]


Done!
Starting Experiment with expected runtime: 3.6 s.


100%|██████████| 2/2 [00:05<00:00,  2.82s/it]


Done!
Starting Experiment with expected runtime: 3.6 s.


100%|██████████| 2/2 [00:05<00:00,  2.81s/it]

Done!


## **Experiment B**

In [5]:
trials_at_each_freq = CONFIG.B_trials_at_each_freq
num_of_freqs = CONFIG.B_num_of_freqs
pad = 10    # arbitrary padding of freq range around theoretical target freq
# make padding higher than in Exp A because temp changes resonance position

# pick only the first resonance frequency to examine
freq = 125

max_voltage = CONFIG.B_max_voltage
num_voltages = CONFIG.B_num_voltages
# start at high voltage
voltage_series = np.flip(np.linspace(0, max_voltage, num_voltages))

loc_prefix = CONFIG.loc_prefix

locs = []

print(f"Analysing {len(voltage_series)} voltages.")
for i, voltage in enumerate(voltage_series):
    loc = f"{loc_prefix}/Experiment_B/voltage{i}_data"
    freqs_series = np.geomspace(freq - pad, freq + pad, num=num_of_freqs)
    expB = ExperimentB(loc, freqs_series, voltage, trials=trials_at_each_freq)
    
    expB.execute()
    locs.append(expB.loc)

expB.devices.reset()

Analysing 10 voltages.
Finished setting voltage: 1.0 V. Current temp: 69.980430186 C.                       
Starting Experiment with expected runtime: 3960.0 s.


100%|██████████| 220/220 [1:11:33<00:00, 19.51s/it]


Finished setting voltage: 0.8888888888888888 V. Current temp: 72.00790907400004 C.                       
Starting Experiment with expected runtime: 3960.0 s.


100%|██████████| 220/220 [1:11:33<00:00, 19.51s/it]


Finished setting voltage: 0.7777777777777777 V. Current temp: 73.80723177 C.                       
Starting Experiment with expected runtime: 3960.0 s.


...

100%|██████████| 220/220 [1:11:33<00:00, 19.51s/it]


Finished setting voltage: 0.1111111111111111 V. Current temp: 42.695023770000034 C.                       
Starting Experiment with expected runtime: 3960.0 s.


100%|██████████| 220/220 [1:11:33<00:00, 19.51s/it]


Finished setting voltage: 0.0 V. Current temp: 38.82933192599999 C.                       
Starting Experiment with expected runtime: 3960.0 s.


100%|██████████| 220/220 [1:11:33<00:00, 19.51s/it]


In [6]:
devices = Devices()

devices.reset()